In [ ]:
import yfinance as yf
import pandas as pd
import numpy as np

tickers = ['PLY.AX', 'LAU.AX', 'COS.AX', 'ANG.AX', 'VVA.AX',
           'WTC.AX', 'AUB.AX', 'TLX.AX', 'DUG.AX']

data = yf.download(tickers, period='5y', interval='1d')['Close']

daily_log_returns = np.log(data / data.shift(1))

monthly_prices = data.resample('ME').last()
monthly_returns = np.log(monthly_prices / monthly_prices.shift(1))

monthly_volatility = daily_log_returns.resample('ME').std().iloc[1:]

result = pd.concat({
    'monthly_return': monthly_returns,
    'monthly_vol': monthly_volatility
}, axis=1).dropna()

print(monthly_returns.shape, monthly_volatility.shape)

[*********************100%***********************]  9 of 9 completed

(60, 9) (60, 9)


In [24]:
import statsmodels.api as sm

results = {}

for stock in monthly_returns.columns:
    y = monthly_returns[stock]
    x = monthly_volatility[stock]
    
    x = sm.add_constant(x)
    
    model = sm.OLS(y, x).fit()
    results[stock] = model
    
    print(f"\n{stock}")
    print(model.summary())


ANG.AX
                            OLS Regression Results                            
Dep. Variable:                 ANG.AX   R-squared:                       0.052
Model:                            OLS   Adj. R-squared:                  0.036
Method:                 Least Squares   F-statistic:                     3.207
Date:                Sat, 11 Apr 2026   Prob (F-statistic):             0.0785
Time:                        09:21:05   Log-Likelihood:                 36.827
No. Observations:                  60   AIC:                            -69.65
Df Residuals:                      58   BIC:                            -65.47
Df Model:                           1                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
const          0.0806      0.046      1.750 

In [49]:
momentum = monthly_returns.rolling(window=3).sum().shift(1)
momentum = momentum.dropna()

volume_data = yf.download(tickers, period='5y', interval='1d')['Volume']
liquidity = np.log(volume_data.resample('ME').sum())

market_data = yf.download('^AXJO', period='5y', interval='1d')['Close']
market_data = market_data.resample('ME').last()
market_ret = np.log(market_data/market_data.shift(1)).dropna()

market_ret_df = pd.DataFrame(
    np.tile(market_ret.values.reshape(-1, 1), (1, len(tickers))),
    index=market_ret.index,
    columns=tickers
)

df = pd.concat({
    'return': monthly_returns,
    'vol': monthly_volatility,
    'momentum': momentum,
    'liquidity': liquidity,
    'market performance': market_ret_df
}, axis=1).dropna()


[*********************100%***********************]  9 of 9 completed
[*********************100%***********************]  1 of 1 completed
C:\Users\suhit\AppData\Local\Temp\ipykernel_16860\1092102379.py:17: Pandas4Warning: Sorting by default when concatenating all DatetimeIndex is deprecated.  In the future, pandas will respect the default of `sort=False`. Specify `sort=True` or `sort=False` to silence this message. If you see this warnings when not directly calling concat, report a bug to pandas.
  df = pd.concat({


In [50]:
print(market_ret_df.columns)

Index(['PLY.AX', 'LAU.AX', 'COS.AX', 'ANG.AX', 'VVA.AX', 'WTC.AX', 'AUB.AX',
       'TLX.AX', 'DUG.AX'],
      dtype='str')


In [57]:
betas = pd.DataFrame()
r_sq = {}

for stock in tickers:
    
    y = df['return'][stock]
    
    X = pd.concat([
        df['vol'][stock],
        df['momentum'][stock],
        df['liquidity'][stock],
        df[('market performance', stock)]
    ], axis=1)
    
    X.columns = ['vol', 'momentum', 'liquidity', 'market performance']
    X = sm.add_constant(X)
    
    model = sm.OLS(y, X).fit(cov_type="HAC", cov_kwds={'maxlags':5})
    
    betas[stock] = model.params
    r_sq[stock] = model.rsquared

betas = betas.T
r_sq = pd.Series(r_sq)

print(betas)
print(f"\n{r_sq}")

           const       vol  momentum  liquidity  market performance
PLY.AX  0.368571 -2.115895  0.124868  -0.017309            1.620022
LAU.AX  0.029546  3.842060 -0.032770  -0.007113            0.795084
COS.AX  0.377160 -3.422753 -0.087364  -0.021295            0.532046
ANG.AX  0.299415 -1.625971  0.003589  -0.014623            0.916576
VVA.AX -0.465641  2.299725 -0.031259   0.028070            0.594939
WTC.AX  1.199202 -0.643015 -0.003355  -0.071958            1.712497
AUB.AX  0.643899 -3.249098 -0.147594  -0.037550            0.592632
TLX.AX  1.561266 -0.770643 -0.069205  -0.089021            0.958836
DUG.AX -0.217805  7.171795  0.215651  -0.001530            0.402374

PLY.AX    0.191133
LAU.AX    0.146910
COS.AX    0.344196
ANG.AX    0.131768
VVA.AX    0.135841
WTC.AX    0.301300
AUB.AX    0.295429
TLX.AX    0.122164
DUG.AX    0.246242
dtype: float64


In [61]:
factor_returns = {}

for month in df.index:
    y = df.loc[month, 'return']   
    
    X = pd.concat([
        df.loc[month, 'vol'],
        df.loc[month, 'momentum'],
        df.loc[month, 'liquidity']
    ], axis=1)
    
    X.columns = ['vol', 'momentum', 'liquidity']
    X = sm.add_constant(X)
    
    model = sm.OLS(y, X).fit()
    factor_returns[month] = model.params

factor_returns = pd.DataFrame(factor_returns).T

print(factor_returns)

               const        vol  momentum  liquidity
2021-08-31 -1.003603   6.119416  0.055689   0.054348
2021-09-30 -0.174375   8.282425 -0.053515  -0.000926
2021-10-31 -0.689897  16.014411 -0.118222   0.019539
2021-11-30  0.610557  -1.024024  0.256528  -0.040651
2021-12-31 -0.552642   1.369756  0.113397   0.037177
2022-01-31  0.022354   3.712723 -0.139821  -0.014757
2022-02-28  0.572042   2.373116 -0.115984  -0.044717
2022-03-31 -0.096303  -5.663850  0.156375   0.018998
2022-04-30  0.317203   1.603199 -0.137872  -0.026443
2022-05-31  0.078029  -6.750568 -0.665810   0.001749
2022-06-30 -0.180638   3.027059  1.310019   0.005556
2022-07-31 -0.711124   3.414003  0.110873   0.051913
2022-08-31 -1.202432  -0.361812 -0.598671   0.083339
2022-09-30  0.553339  -1.914019  0.276168  -0.042407
2022-10-31 -1.654117   4.657562 -0.001444   0.101825
2022-11-30  0.314423  -1.179618 -0.303234  -0.012670
2022-12-31 -0.122376   7.175586  0.415289  -0.005007
2023-01-31  0.524017  -0.889197 -0.169081  -0.